In [ ]:
# CELL 1 — imports
import json
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from src.features import build_modeling_frame, TARGET

In [ ]:
# CELL 2 — load
df = pd.read_parquet('../data/processed/train.parquet')
X, y, cat_cols = build_modeling_frame(df)
print('X:', X.shape, '  y:', y.shape, '  cats:', len(cat_cols))

In [ ]:
# CELL 3 — 5-fold CV with LightGBM
kf = KFold(n_splits=5, shuffle=True, random_state=42)
maes, rmses, r2s = [], [], []
oof_pred = np.zeros(len(y))
for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
    Xtr, ytr = X.iloc[tr_idx], y.iloc[tr_idx]
    Xva, yva = X.iloc[va_idx], y.iloc[va_idx]
    model = lgb.LGBMRegressor(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=20,
        feature_fraction=0.9,
        bagging_fraction=0.9,
        bagging_freq=5,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(Xtr, ytr,
              categorical_feature=cat_cols,
              eval_set=[(Xva, yva)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = model.predict(Xva)
    oof_pred[va_idx] = pred
    mae  = mean_absolute_error(yva, pred)
    rmse = root_mean_squared_error(yva, pred)
    r2   = r2_score(yva, pred)
    maes.append(mae); rmses.append(rmse); r2s.append(r2)
    print(f'Fold {fold}:  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}')
print(f'\nMean MAE  = {np.mean(maes):.3f} ± {np.std(maes):.3f}')
print(f'Mean RMSE = {np.mean(rmses):.3f}')
print(f'Mean R2   = {np.mean(r2s):.3f}')

In [ ]:
# CELL 4 — save scores for P3
scores = {
    'fold_mae':  [float(x) for x in maes],
    'fold_rmse': [float(x) for x in rmses],
    'fold_r2':   [float(x) for x in r2s],
    'mae_mean':  float(np.mean(maes)),
    'mae_std':   float(np.std(maes)),
    'rmse_mean': float(np.mean(rmses)),
    'r2_mean':   float(np.mean(r2s)),
    'n_features': int(X.shape[1]),
    'n_rows': int(X.shape[0]),
}
with open('../reports/scores_v1.json', 'w') as f:
    json.dump(scores, f, indent=2)
print('Saved reports/scores_v1.json')